# Bronze Ingestion - JPL Horizons

Ingests the raw NASA/JPL Horizons response for the configured Artemis II mission window into the Bronze Delta table. Table creation is handled by `notebooks/00_setup/01_create_bronze_tables.py.ipynb`; this notebook only runs ingestion and verifies the result.


In [0]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "orion" / "config.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break


In [0]:
from src.orion.config import JPL_HORIZONS_BRONZE_COLUMNS
from src.orion.ingestion.jpl_horizons import ingest_jpl_horizons, raw_jpl_horizons_table_name

target_table = raw_jpl_horizons_table_name()
print(f"Target table: {target_table}")
print(f"Expected columns: {', '.join(JPL_HORIZONS_BRONZE_COLUMNS)}")


In [0]:
result = ingest_jpl_horizons(spark, target_table=target_table)

print(f"Run ID: {result['ingestion_run_id']}")
print(f"Status code: {result['response_status_code']}")
print(f"Records written: {result['records_written']}")
print(f"Response hash: {result['response_hash']}")
print(result["request_url"])


In [0]:
display(
    spark.sql(f"""
        SELECT
            ingestion_run_id,
            source_system,
            source_endpoint,
            response_status_code,
            length(response_body) AS response_body_length,
            response_hash,
            ingested_at,
            ingested_date,
            mission_name
        FROM {target_table}
        ORDER BY ingested_at DESC
        LIMIT 10
    """)
)
